In [ ]:
import os 

import gdown
from zipfile import ZipFile
from io import BytesIO
from PIL import Image
 
from transformers import CLIPProcessor, CLIPModel

import torch
import ipyplot

In [ ]:
LOCAL_ZIP_NAME = 'travel_images.zip'
PATH_TO_ZIP = os.path.join('./images/', LOCAL_ZIP_NAME)
URL = 'https://drive.google.com/uc?id=1f2he9Ipx_SI_aAmrpEhfjKfrm7POuJti'

semantic_search_phrase = "Sonnenblumenfeld"

In [ ]:
def download_images():
    """Download images from Google Drive."""
    if os.path.exists(PATH_TO_ZIP):
        print(f"Images already downloaded. Uisng {PATH_TO_ZIP}.")
    else: 
        if os.path.exists('./images/') is False:
            os.makedirs('./images/')
        print(f"Downloading images from {URL} to {PATH_TO_ZIP}.")
        gdown.download(URL, PATH_TO_ZIP, quiet=False)
        
    images = []
    with ZipFile(PATH_TO_ZIP, 'r') as zip_ref:
        for file_name in zip_ref.namelist():
            if file_name.lower().endswith(('png', 'jpg', 'jpeg')):
                with zip_ref.open(file_name) as file:
                    image = Image.open(BytesIO(file.read())).convert('RGB')
                    images.append(image)

    return images

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
images = download_images()

# Compute similarities

In [ ]:
with torch.no_grad():
    inputs = processor(text=[semantic_search_phrase], 
                    images=images, return_tensors="pt", padding=True)
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities

In [ ]:
values, indices = logits_per_image.squeeze().topk(3) # Top-3

top_images, top_scores = [], []

for score, index in zip(values, indices):
    top_images.append(images[int(index.numpy())])
    score = score.numpy().tolist()
    top_scores.append(round(score, 3))

In [ ]:
print (f"Scores: {top_scores}")
ipyplot.plot_images(top_images, img_width=300)